# Reshaping — pivot, melt, stack/unstack, crosstab

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from _data import load_accidents

df = load_accidents()
df.shape

## 1. `pivot_table` — données longues → large

`pivot_table` transforme des données en **format long** (une observation par ligne)
en **format large** (une variable devient des colonnes).

Cas canonique : accidents par mois × département → matrice.

```
Format long          →   Format large (pivot)
mois  dep  nb             dep →  75   69   13  ...
  1    75   12         mois ↓
  1    69    8              1    12    8    6
  2    75   15              2    15   11    9
  2    69   11              ...
  ...
```

In [ ]:
# Nombre d'accidents par mois × département — top 10 deps pour la lisibilité
top_deps = df["dep"].value_counts().nlargest(8).index.tolist()

pivot = (
    df
    .query("dep in @top_deps")
    .pivot_table(
        index="mois",
        columns="dep",
        values="Num_Acc",
        aggfunc="count",
        fill_value=0,
    )
)
pivot

In [ ]:
# Paramètres clés de pivot_table
# aggfunc : que faire quand plusieurs lignes tombent dans la même case
# fill_value : valeur pour les cases vides (NaN par défaut)
# margins=True : totaux ligne/colonne

(
    df
    .query("dep in @top_deps")
    .pivot_table(
        index="mois",
        columns="dep",
        values="Num_Acc",
        aggfunc="count",
        fill_value=0,
        margins=True,
        margins_name="Total",
    )
)

In [ ]:
# pivot_table avec plusieurs valeurs — produit un MultiIndex en colonnes
(
    df
    .query("dep in ['75', '69', '13']")
    .pivot_table(
        index="mois",
        columns="dep",
        values=["Num_Acc", "vma"],
        aggfunc={"Num_Acc": "count", "vma": "mean"},
    )
    .round(0)
    .head(4)
)

### `pivot` vs `pivot_table`

- **`pivot_table`** : agrège les doublons avec `aggfunc`. Cas général, toujours safe.
- **`pivot`** : suppose qu'il n'y a **qu'une seule valeur** par combinaison (index, column). Plante si doublons.

In [ ]:
# pivot fonctionne seulement si index + columns identifient une ligne unique
# Ici on crée un agrégat propre d'abord
df_agg = (
    df
    .query("dep in ['75', '69', '13']")
    .groupby(["mois", "dep"])["Num_Acc"]
    .count()
    .reset_index(name="nb_accidents")
)

# Maintenant on peut utiliser pivot (1 seule valeur par mois×dep)
df_agg.pivot(index="mois", columns="dep", values="nb_accidents")

## 2. `melt` — données larges → longues

`melt` est l'opération inverse de `pivot` :
plusieurs colonnes deviennent les valeurs d'une seule colonne "variable".

In [ ]:
# On part du pivot créé plus haut et on le remet en format long
pivot_reset = pivot.reset_index()
print("Format large :", pivot_reset.shape)
print(pivot_reset.head(3))

In [ ]:
long = pivot_reset.melt(
    id_vars="mois",       # colonnes à garder en place
    var_name="dep",       # nom de la colonne qui va recevoir les anciens noms de colonnes
    value_name="nb_accidents",  # nom de la colonne de valeurs
)
print("Format long :", long.shape)
long.head(6)

In [ ]:
# Cas d'usage réel : un DataFrame avec des colonnes secu1, secu2, secu3
# (équipements de sécurité) qu'on veut analyser ensemble
from _data import load_accidents_usagers
usagers = load_accidents_usagers()

securite_long = (
    usagers[["id_usager", "secu1", "secu2", "secu3"]]
    .melt(
        id_vars="id_usager",
        var_name="rang_equipement",
        value_name="code_equipement",
    )
    .query("code_equipement > 0")  # 0 = non renseigné
)
print(securite_long.shape)
securite_long.head(5)

## 3. `stack` / `unstack` — travailler avec les MultiIndex

`stack` fait pivoter les colonnes en lignes (vers un index hiérarchique).
`unstack` fait l'inverse — c'est l'équivalent de `pivot` sur un MultiIndex.

In [ ]:
# Groupby à deux niveaux → MultiIndex naturel
multi = (
    df
    .query("dep in ['75', '69', '13']")
    .groupby(["mois", "dep"])["Num_Acc"]
    .count()
)
print("MultiIndex (mois, dep) :")
multi.head(6)

In [ ]:
# unstack : le niveau dep devient des colonnes → format large
multi.unstack(level="dep")

In [ ]:
# stack : l'opération inverse — les colonnes dep retournent en lignes
large = multi.unstack(level="dep")
large.stack(future_stack=True).head(6)  # future_stack=True : comportement pandas 3.0

## 4. `crosstab` — tableau de contingence

`crosstab` est un raccourci pour compter les co-occurrences de deux variables catégorielles.
Idéal pour explorer les relations entre variables avant de modéliser.

In [ ]:
# Distribution des accidents par conditions atmosphériques × luminosité
# atm : 1=normale, 2=pluie légère, 3=pluie forte, 4=neige, 5=brouillard...
# lum : 1=jour, 2=crépuscule, 3=nuit éclairée, 4=nuit non éclairée
pd.crosstab(df["lum"], df["atm"])

In [ ]:
# normalize : proportion au lieu des effectifs
# normalize="index" : chaque ligne somme à 1
pd.crosstab(df["lum"], df["atm"], normalize="index").round(2)

In [ ]:
# Avec margins=True : totaux
pd.crosstab(df["lum"], df["col"], margins=True, margins_name="Total")

In [ ]:
# crosstab avec une valeur agrégée (comme pivot_table)
# Ici : vitesse maximale autorisée médiane par luminosité × type de voie
pd.crosstab(
    index=df["lum"],
    columns=df["catr"],
    values=df["vma"],
    aggfunc="median",
).round(0).dropna(how="all")

## Récapitulatif

| Transformation | Outil | Direction |
|---|---|---|
| Long → Large (agrégation) | `pivot_table(index, columns, values, aggfunc)` | Colonnes prolifèrent |
| Long → Large (sans doublon) | `pivot(index, columns, values)` | Colonnes prolifèrent |
| Large → Long | `melt(id_vars, var_name, value_name)` | Lignes prolifèrent |
| MultiIndex colonnes → lignes | `stack()` | Lignes prolifèrent |
| MultiIndex lignes → colonnes | `unstack()` | Colonnes prolifèrent |
| Tableau de contingence | `pd.crosstab(index, columns)` | Exploration rapide |